# RAG Chatbot for Large Language Model Research Papers
**Project:** AI-Client-Project-5 — Retrieval-Augmented Generation (RAG) Chatbot  
**Student:** Kamal Khan  **Course:** AIE  **Team ID:** PTID-AIE-APR-26-11987

This notebook contains the complete project: problem description, RAG pipeline, saved vector store, chatbot interface, RAGAS evaluation and documentation.

**Contents**
1. Problem domain description
2. Architecture
3. Setup
4. Load and chunk the documents
5. Embeddings and the ChromaDB vector store
6. Retrieval quality check
7. Generation: LLM, prompt and `ask()`
8. Demo: grounded answers and refusals
9. Chatbot interface (Streamlit app and CLI)
10. Evaluation with RAGAS
11. Analysis, limitations and future work
12. How to run

## 1. Problem domain description

**Who is the chatbot for?**  
Students, beginners and AI practitioners who want quick, reliable answers about how large language models work without reading every research paper in full.

**What documents does it use?**  
Eight foundational LLM research papers from arXiv, as PDFs: *Attention Is All You Need*, *BERT*, *Language Models are Few-Shot Learners (GPT-3)*, *Retrieval-Augmented Generation (RAG)*, *Chain-of-Thought Prompting*, *LoRA*, *LLaMA*, and *A Survey of Large Language Models* (365 pages in total).

**What problem does it solve?**  
Research papers are long, technical and hard to search. A general-purpose LLM asked about them may hallucinate details such as wrong numbers or invented claims. This chatbot retrieves the relevant passages from the actual papers and generates answers **only** from that content, so responses are accurate and traceable to a source (file and page). Questions the papers cannot answer are refused instead of guessed.

## 2. Architecture

```
PHASE 1 - BUILD THE INDEX (run once)
 ┌────────────┐   ┌──────────────┐   ┌──────────────────┐   ┌─────────────────┐
 │ 8 PDF      │──▶│ Chunks       │──▶│ Embeddings       │──▶│ ChromaDB        │
 │ papers     │   │ 1000 chars,  │   │ all-MiniLM-L6-v2 │   │ vector store    │
 │ (DATA/)    │   │ 150 overlap  │   │ (384-d vectors)  │   │ (chroma_db/)    │
 └────────────┘   └──────────────┘   └──────────────────┘   └────────┬────────┘
                                                                     │ similarity search
PHASE 2 - ANSWER A QUESTION (every query)                            ▼
 ┌────────────┐   ┌──────────────────────────┐   ┌─────────────┐   ┌──────────────┐
 │ User       │──▶│ Retriever                │──▶│ Prompt +    │──▶│ Answer       │
 │ question   │   │ embed question, top-k=6  │   │ Groq LLM    │   │ + sources    │
 └────────────┘   └──────────────────────────┘   └─────────────┘   └──────────────┘
```

| Component | Choice |
|---|---|
| Orchestration | LangChain |
| LLM | Groq API (`openai/gpt-oss` models) |
| Embeddings | `sentence-transformers/all-MiniLM-L6-v2` (local, CPU) |
| Vector store | ChromaDB (persisted to `chroma_db/`) |
| UI | Streamlit (`app.py`) and a CLI chat function in this notebook |
| Evaluation | RAGAS (faithfulness and answer relevancy) |

## 3. Setup
Run this notebook with the kernel set to the project's virtual environment, with the notebook saved in the project folder (next to `DATA/`). A Groq API key must be in a `.env` file as `GROQ_API_KEY=...`.

In [1]:
import os, sys, types, time, warnings
from pathlib import Path
from dotenv import load_dotenv

warnings.filterwarnings("ignore")

BASE_DIR = Path.cwd()                 # the notebook lives in the project folder
DATA_DIR = BASE_DIR / "DATA"          # the 8 PDF papers
DB_DIR   = BASE_DIR / "chroma_db"     # saved vector store

# ---- Settings used in this project ----
CHUNK_SIZE, CHUNK_OVERLAP = 1000, 150
TOP_K = 6                             # chosen after comparing k=4 and k=6 (section 10)
EMBED_MODEL = "sentence-transformers/all-MiniLM-L6-v2"
LLM_MODEL = "openai/gpt-oss-20b"      # the saved RAGAS scores in section 10 were produced with openai/gpt-oss-120b

load_dotenv(BASE_DIR / ".env")
assert os.getenv("GROQ_API_KEY"), "GROQ_API_KEY not found. Create a .env file with GROQ_API_KEY=your_key"
assert DATA_DIR.exists(), f"DATA folder not found at {DATA_DIR}"
print("Project folder:", BASE_DIR)
print("API key found: yes (value not shown)")
print("PDF files in DATA/:", len(list(DATA_DIR.glob('*.pdf'))))

Project folder: C:\RAG_Chatbot
API key found: yes (value not shown)
PDF files in DATA/: 8


## 4. Load and chunk the documents
The PDFs are loaded page by page, then split into overlapping chunks so that a sentence cut at a boundary still appears whole in a neighbouring chunk.

In [2]:
from langchain_community.document_loaders import PyPDFDirectoryLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

docs = PyPDFDirectoryLoader(str(DATA_DIR)).load()
pdf_names = sorted({Path(d.metadata["source"]).name for d in docs})
print(f"Loaded {len(docs)} pages from {len(pdf_names)} PDFs:")
for n in pdf_names:
    print("  -", n)

splitter = RecursiveCharacterTextSplitter(chunk_size=CHUNK_SIZE, chunk_overlap=CHUNK_OVERLAP)
chunks = splitter.split_documents(docs)
print(f"\nCreated {len(chunks)} chunks (size={CHUNK_SIZE}, overlap={CHUNK_OVERLAP})")

sample = chunks[100]
print("\nExample chunk ->", Path(sample.metadata["source"]).name, "| page", sample.metadata.get("page", 0) + 1)
print(sample.page_content[:400])

Loaded 365 pages from 8 PDFs:
  - Attention Is All You Need.pdf
  - BERT.pdf
  - Chain_of_thought.pdf
  - LLama.pdf
  - LORA.pdf
  - RAG.pdf
  - gpt3.pdf
  - llm_survey.pdf

Created 1970 chunks (size=1000, overlap=150)

Example chunk -> BERT.pdf | page 10
guistics.
Ciprian Chelba, Tomas Mikolov, Mike Schuster, Qi Ge,
Thorsten Brants, Phillipp Koehn, and Tony Robin-
son. 2013. One billion word benchmark for measur-
ing progress in statistical language modeling. arXiv
preprint arXiv:1312.3005.
Z. Chen, H. Zhang, X. Zhang, and L. Zhao. 2018.
Quora question pairs.
Christopher Clark and Matt Gardner. 2018. Simple
and effective multi-paragraph reading co


## 5. Embeddings and the ChromaDB vector store
Each chunk is converted to a vector with a local sentence-transformer model and stored in ChromaDB. The store is saved to disk, so the notebook **reuses it if it already exists** and builds it only once (reproducible, no duplicate chunks).

In [3]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma

embeddings = HuggingFaceEmbeddings(model_name=EMBED_MODEL)

if DB_DIR.exists() and any(DB_DIR.iterdir()):
    vectordb = Chroma(persist_directory=str(DB_DIR), embedding_function=embeddings)
    print("Loaded the existing vector store from", DB_DIR)
else:
    vectordb = Chroma.from_documents(documents=chunks, embedding=embeddings,
                                     persist_directory=str(DB_DIR))
    print("Built and saved a new vector store at", DB_DIR)

print("Vectors in the store:", vectordb._collection.count())

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2118.43it/s]


Loaded the existing vector store from C:\RAG_Chatbot\chroma_db
Vectors in the store: 1970


## 6. Retrieval quality check
Before generating any answer, look at *which chunks* the retriever returns for a question. Good retrieval is the foundation of a good RAG answer.

In [4]:
retriever = vectordb.as_retriever(search_kwargs={"k": TOP_K})

question = "What does LoRA freeze and what does it train?"
print("Question:", question, "\n")
for i, d in enumerate(retriever.invoke(question), 1):
    src = Path(d.metadata["source"]).name
    print(f"[{i}] {src} | page {d.metadata.get('page', 0) + 1}")
    print("    " + d.page_content[:220].replace("\n", " ") + " ...\n")

Question: What does LoRA freeze and what does it train? 

[1] LORA.pdf | page 4
    a subset of the pre-trained parameters. LoRA takes a step further and does not require the accumu- lated gradient update to weight matrices to have full-rank during adaptation. This means that when applying LoRA to all w ...

[2] LORA.pdf | page 21
    MultiNLI, the combination of LoRA+PE doesn’t perform better than LoRA, possibly because LoRA on its own already achieves performance comparable to the human baseline. Secondly, we notice that LoRA+PL performs slightly wo ...

[3] llm_survey.pdf | page 44
    44 5.3.2 Parameter-Efficient Fine-Tuning on LLMs With the rising of LLMs, efficient tuning has attracted increasing research attention for developing a more lightweight adaptation approach in downstream tasks. In particu ...

[4] llm_survey.pdf | page 87
    or by finding a proper low-rank initizalization for LoRA fine-tuning [1017]. Secondly, it is flexible to support task- specific or goal-specific 

## 7. Generation: LLM, grounded prompt and `ask()`
The prompt instructs the model to answer **only** from the retrieved context and to say it does not have enough information otherwise. This is what prevents hallucination.

In [5]:
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate

llm = ChatGroq(model=LLM_MODEL, temperature=0)

prompt = ChatPromptTemplate.from_template(
    """You are a helpful assistant that answers questions about Large Language Model research papers.
Answer ONLY using the context below.
If the answer is not in the context, say exactly: "I don't have enough information in the documents to answer that."

Context:
{context}

Question: {question}

Answer:"""
)

def ask(question: str):
    """Retrieve the top-k chunks, then generate an answer grounded in them."""
    docs = retriever.invoke(question)
    context = "\n\n".join(d.page_content for d in docs)
    answer = (prompt | llm).invoke({"context": context, "question": question}).content
    return answer, docs

def show(question: str):
    answer, docs = ask(question)
    print("QUESTION:", question)
    print("ANSWER  :", answer)
    print("SOURCES :", "; ".join(sorted({f"{Path(d.metadata['source']).name} p.{d.metadata.get('page', 0) + 1}" for d in docs})))
    print("-" * 100)

print("Model:", LLM_MODEL, "| top-k:", TOP_K)

Model: openai/gpt-oss-20b | top-k: 6


## 8. Demo: grounded answers and refusals
Three in-scope questions (answered from the papers, with sources) and two out-of-scope questions (which the chatbot must refuse).

In [6]:
in_scope = [
    "What does LoRA freeze and what does it train?",
    "What are the two components of a RAG model?",
    "What is chain-of-thought prompting?",
]
out_of_scope = [
    "Who won the 2022 FIFA World Cup?",
    "What is Ollama?",
]

print("=== IN-SCOPE QUESTIONS ===\n")
for q in in_scope:
    show(q)
    time.sleep(1)

print("\n=== OUT-OF-SCOPE QUESTIONS (should be refused) ===\n")
for q in out_of_scope:
    show(q)
    time.sleep(1)

=== IN-SCOPE QUESTIONS ===

QUESTION: What does LoRA freeze and what does it train?
ANSWER  : LoRA freezes the original pre‑trained weight matrices (W₀) and only learns a small low‑rank update (the product BA) together with the model’s biases.
SOURCES : LORA.pdf p.10; LORA.pdf p.12; LORA.pdf p.21; LORA.pdf p.4; llm_survey.pdf p.44; llm_survey.pdf p.87
----------------------------------------------------------------------------------------------------
QUESTION: What are the two components of a RAG model?
ANSWER  : The RAG model consists of two parts: a **non‑parametric memory (retriever)** that fetches relevant documents, and a **parametric memory (generator)** that produces the final text.
SOURCES : RAG.pdf p.10; RAG.pdf p.2; RAG.pdf p.6; RAG.pdf p.7; RAG.pdf p.8
----------------------------------------------------------------------------------------------------
QUESTION: What is chain-of-thought prompting?
ANSWER  : Chain‑of‑thought prompting is a few‑shot prompting technique in which

## 9. Chatbot interface

**Streamlit app (`app.py`)** - a chat UI with message history and an expandable *Sources* panel for every answer. It is launched separately from a terminal with `streamlit run app.py` (see section 12). Its source code:

```python
from pathlib import Path
import streamlit as st
from src.rag_chain import ask

st.set_page_config(page_title="LLM Papers RAG Chatbot", page_icon="📚")
st.title("📚 LLM Research Papers Chatbot")
st.caption("Answers come only from the 8 papers in the knowledge base.")

if "history" not in st.session_state:
    st.session_state.history = []

for msg in st.session_state.history:
    with st.chat_message(msg["role"]):
        st.write(msg["content"])

if question := st.chat_input("Ask a question about the papers..."):
    st.session_state.history.append({"role": "user", "content": question})
    with st.chat_message("user"):
        st.write(question)
    with st.chat_message("assistant"):
        with st.spinner("Searching the papers..."):
            answer, docs = ask(question)
        st.write(answer)
        with st.expander("Sources"):
            for d in docs:
                st.markdown(f"**{Path(d.metadata.get('source','')).name}**, page {d.metadata.get('page', 0) + 1}")
                st.caption(d.page_content[:300] + "...")
    st.session_state.history.append({"role": "assistant", "content": answer})
```

**CLI chatbot (works directly in this notebook)** - run the cell below and type questions. Type `exit` to stop.

In [7]:
def chat():
    """Simple command-line chatbot. Type 'exit' to quit."""
    print("RAG chatbot ready. Ask about the LLM papers (type 'exit' to stop).")
    while True:
        q = input("\nYou: ").strip()
        if q.lower() in {"exit", "quit", ""}:
            print("Goodbye!")
            break
        answer, docs = ask(q)
        print("Bot:", answer)
        print("Sources:", "; ".join(sorted({f"{Path(d.metadata['source']).name} p.{d.metadata.get('page', 0) + 1}" for d in docs})))

# chat()   # <- uncomment and run for an interactive session

## 10. Evaluation with RAGAS

**Method**
- **Test set:** 10 questions with reference answers written from the papers, plus the 2 out-of-scope questions from section 8.
- **Metrics (RAGAS):** *faithfulness* - are the claims in the answer supported by the retrieved chunks (hallucination check)? *Answer relevancy* - does the answer address the question asked?
- **Experiment:** one variable changed, the number of retrieved chunks **k** (4 vs 6). Everything else was identical.
- **Judge model:** `openai/gpt-oss-120b` (via Groq) with the local MiniLM embeddings.

The RAGAS scoring makes many LLM calls and is limited by the free-tier daily token quota, so the scores below are **loaded from the result files saved by the evaluation runs** (`eval_results_k4.csv`, `eval_results_k6.csv`). The code that produces them is included at the end of this section (switch `RUN_RAGAS = True` to re-run it).

In [8]:
QA = [
    {"q": "What does the Transformer use instead of recurrence and convolution?",
     "ref": "It relies entirely on attention mechanisms (self-attention), which allows much more parallelization."},
    {"q": "What is masked language modeling in BERT?",
     "ref": "Some input tokens are randomly masked and the model is trained to predict the masked tokens using context from both directions."},
    {"q": "What is few-shot learning in GPT-3?",
     "ref": "The model is given a few task demonstrations in its context at inference time, without any gradient updates or fine-tuning."},
    {"q": "What are the two components of a RAG model?",
     "ref": "A retriever that fetches relevant documents and a sequence-to-sequence generator that produces the answer conditioned on them."},
    {"q": "What are the two variants of RAG?",
     "ref": "RAG-Sequence, which uses the same retrieved document for the whole output, and RAG-Token, which can use different documents per token."},
    {"q": "What is chain-of-thought prompting?",
     "ref": "Prompting a language model with examples that include intermediate reasoning steps, so it produces step-by-step reasoning before the final answer."},
    {"q": "What does LoRA freeze and what does it train?",
     "ref": "LoRA freezes the pre-trained model weights and trains small low-rank decomposition matrices injected into the layers."},
    {"q": "How many parameters does GPT-3 have?",
     "ref": "GPT-3 has 175 billion parameters."},
    {"q": "Which paper introduced multi-head attention?",
     "ref": "The paper 'Attention Is All You Need', which introduced the Transformer."},
    {"q": "What data was LLaMA trained on?",
     "ref": "LLaMA was trained only on publicly available datasets."},
]
import pandas as pd
pd.set_option("display.width", 220); pd.set_option("display.max_colwidth", 90)
display(pd.DataFrame(QA).rename(columns={"q": "question", "ref": "reference answer"}))

,question,reference answer
0,What does the Transformer use instead of recurrence and convolution?,"It relies entirely on attention mechanisms (self-attention), which allows much more pa..."
1,What is masked language modeling in BERT?,Some input tokens are randomly masked and the model is trained to predict the masked t...
2,What is few-shot learning in GPT-3?,"The model is given a few task demonstrations in its context at inference time, without..."
3,What are the two components of a RAG model?,A retriever that fetches relevant documents and a sequence-to-sequence generator that ...
4,What are the two variants of RAG?,"RAG-Sequence, which uses the same retrieved document for the whole output, and RAG-Tok..."
5,What is chain-of-thought prompting?,"Prompting a language model with examples that include intermediate reasoning steps, so..."
6,What does LoRA freeze and what does it train?,LoRA freezes the pre-trained model weights and trains small low-rank decomposition mat...
7,How many parameters does GPT-3 have?,GPT-3 has 175 billion parameters.
8,Which paper introduced multi-head attention?,"The paper 'Attention Is All You Need', which introduced the Transformer."
9,What data was LLaMA trained on?,LLaMA was trained only on publicly available datasets.


In [9]:
k4 = pd.read_csv(BASE_DIR / "eval_results_k4.csv")
k6 = pd.read_csv(BASE_DIR / "eval_results_k6.csv")
cols = ["faithfulness", "answer_relevancy"]

summary = pd.DataFrame({"k = 4": k4[cols].mean(), "k = 6": k6[cols].mean()}).round(3)
summary.loc["questions scored (faithfulness)"] = [int(k4["faithfulness"].notna().sum()), int(k6["faithfulness"].notna().sum())]
print("AVERAGE SCORES")
display(summary)

# Fair comparison: one faithfulness job timed out in the k=6 run, so compare on the same questions
mask = k6["faithfulness"].notna()
print(f"Same {int(mask.sum())} questions -> faithfulness  k=4: {k4.loc[mask,'faithfulness'].mean():.3f}   k=6: {k6.loc[mask,'faithfulness'].mean():.3f}")
print("Timed-out question (no faithfulness score):", k6.loc[~mask, "user_input"].tolist())

print("\nPER-QUESTION SCORES (k = 6, final setting)")
display(k6[["user_input"] + cols].round(2))

AVERAGE SCORES


,k = 4,k = 6
faithfulness,0.702,0.817
answer_relevancy,0.785,0.883
questions scored (faithfulness),10.000,9.000


Same 9 questions -> faithfulness  k=4: 0.669   k=6: 0.817
Timed-out question (no faithfulness score): ['What is masked language modeling in BERT?']

PER-QUESTION SCORES (k = 6, final setting)


,user_input,faithfulness,answer_relevancy
0,What does the Transformer use instead of recurrence and convolution?,0.50,0.96
1,What is masked language modeling in BERT?,NaN,0.81
2,What is few-shot learning in GPT-3?,0.86,0.98
3,What are the two components of a RAG model?,1.00,1.00
4,What are the two variants of RAG?,1.00,1.00
5,What is chain-of-thought prompting?,1.00,0.97
6,What does LoRA freeze and what does it train?,1.00,0.48
7,How many parameters does GPT-3 have?,1.00,1.00
8,Which paper introduced multi-head attention?,0.00,0.94
9,What data was LLaMA trained on?,1.00,0.70


In [10]:
# Optional: re-run the full RAGAS evaluation (uses many API tokens - leave False to use the saved results above)
RUN_RAGAS = False

if RUN_RAGAS:
    # RAGAS 0.4.3 imports a module that newer langchain-community removed; this stub lets it load
    _stub = types.ModuleType("langchain_community.chat_models.vertexai")
    class ChatVertexAI: pass
    _stub.ChatVertexAI = ChatVertexAI
    sys.modules["langchain_community.chat_models.vertexai"] = _stub

    from ragas import evaluate, EvaluationDataset
    from ragas.llms import LangchainLLMWrapper
    from ragas.embeddings import LangchainEmbeddingsWrapper
    from ragas.metrics._faithfulness import Faithfulness
    from ragas.metrics._answer_relevance import ResponseRelevancy
    from ragas.run_config import RunConfig

    rows = []
    for item in QA:
        answer, docs = ask(item["q"])
        rows.append({"user_input": item["q"], "response": answer,
                     "retrieved_contexts": [d.page_content for d in docs],
                     "reference": item["ref"]})
        time.sleep(2)

    result = evaluate(
        EvaluationDataset.from_list(rows),
        metrics=[Faithfulness(), ResponseRelevancy(strictness=1)],
        llm=LangchainLLMWrapper(llm),
        embeddings=LangchainEmbeddingsWrapper(embeddings),
        run_config=RunConfig(max_workers=2, timeout=300),
    )
    df = result.to_pandas()
    print(df.select_dtypes("number").mean().round(3))
else:
    print("RUN_RAGAS is False -> using the saved evaluation results shown above.")

RUN_RAGAS is False -> using the saved evaluation results shown above.


## 11. Analysis, limitations and future work

**Results.** Retrieving 6 chunks instead of 4 raised faithfulness from **0.669 to 0.817** on the same 9 questions (0.702 → 0.817 on the available scores) and answer relevancy from **0.785 to 0.883**. Six of ten questions scored a perfect 1.00 on faithfulness, and the chatbot refused both out-of-scope questions (2/2). **k = 6 was kept as the final setting.**

**Weak spots and likely causes** (hypotheses based on how the metrics work; the `response` and `retrieved_contexts` columns in the result files can be inspected to confirm them):
- *"Which paper introduced multi-head attention?" (faithfulness 0.00)* - this asks for a paper **title**, which is metadata rather than text stated in the body of the papers, so the retrieved chunks probably did not support the claim.
- *"What does the Transformer use instead of recurrence and convolution?" (faithfulness 0.50)* - the answer probably mixed retrieved statements with extra detail from the model's own knowledge, which faithfulness counts as unsupported.
- *"What does LoRA freeze and what does it train?" (relevancy 0.48)* - a two-part question; relevancy is lower when an answer covers only part of it or adds extra material.
- *"What data was LLaMA trained on?" (relevancy 0.70)* - the answer was probably broader than the question needed.
- *Timeout* - one faithfulness job for the BERT question timed out because of free-tier API limits; this reflects infrastructure, not answer quality.

**Limitations**
- Small evaluation: 10 questions and a single run per setting. LLM-judged scores vary a little between runs, so the k = 4 vs k = 6 result is an indication, not proof.
- The same model family generates the answers and judges them, which can favour its own style.
- Fixed-size chunking can cut through tables, equations and section boundaries.
- No conversation memory: every question is answered independently.
- Questions about metadata (such as which paper introduced a concept) are poorly served because retrieval works on body text.
- The free-tier API has daily token limits, which restrict how often the full evaluation can be re-run.

**Future work:** add a re-ranker or hybrid (BM25 + vector) search; attach paper-title metadata to chunks; use section-aware chunking and a layout-aware PDF parser; add conversation memory; expand the test set to 50+ questions and add context precision / recall; compare chunk sizes and embedding models in the same controlled way as k.

## 12. How to run

**Requirements:** Python 3.10+ (developed with 3.13), a free Groq API key from console.groq.com.

```bash
python -m venv venv
venv\Scripts\activate
pip install langchain langchain-community langchain-chroma langchain-huggingface langchain-groq \
            langchain-text-splitters sentence-transformers chromadb pypdf streamlit ragas datasets \
            python-dotenv pandas ipykernel
```

1. Put the 8 PDF papers in a folder named `DATA/`.
2. Create a file named `.env` in the project folder containing `GROQ_API_KEY=your_key_here` (never share this file).
3. Open this notebook from the project folder, select the virtual environment as the kernel, and choose **Run All**. The vector store is built on the first run and reused afterwards.
4. For the Streamlit chat app, save the code in section 9 as `app.py` (with the pipeline in `src/rag_chain.py`) and run `streamlit run app.py`.

**Notes:** the Groq model names change over time - list the models your key can use with `Groq().models.list()` and update `LLM_MODEL` in section 3 if a model is retired. If a daily token limit is reached, switch to `openai/gpt-oss-20b`.